# Two-Phase Bayesian Optimization: Joint LR + Slice Direction

**Dataset**: NYX `temperature` (512×512×512)

- **Phase 1** — Optuna TPE jointly searches `lr ∈ [1e-4, 1e-2]` × `direction ∈ {X, Y, Z}` on a cheap proxy (`TUNE_DEPTH` slices at S=2) in ≤ 20% of the 60 s budget, and outputs `(lr*, d*)`
- **Phase 2 (production)** — trust `(lr*, d*)` and train it on the full volume for the rest of the budget
- **Phase 2 (this notebook = validation)** — re-run *every* Phase-1 combo at full resolution to confirm the BO pick lands at/near the true best

The key fix for *"Phase-1 winner ≠ Phase-2 winner"* is a proxy (full-res patches at S=2, depth-subsampled) that preserves the per-step gradient statistics, so the optimal LR and direction **transfer** to full resolution.

*Implements § Two-Phase Bayesian-Optimized Adaptive Online Learning*

In [ ]:
import random, sys, os, copy, time
from pathlib import Path

import numpy as np
import torch
import matplotlib.pyplot as plt
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)

sys.path.append("/home/sam/Halo_Finder/Final_design/base_script")

from config_io import load_multifield_from_disk
# Force-reload edited modules so re-running this cell picks up bg_stage.py
# changes WITHOUT restarting the kernel (Jupyter caches modules in sys.modules,
# so a plain `from bg_stage import ...` would keep the stale early-stop logic).
import importlib
import bg_stage
importlib.reload(bg_stage)

from experiment import build_bg_only_cfg
from bg_stage import run_bg_inference, train_bg_only, unwrap_bg_model
from bg_shard import pick_bg_h_under_budget

pysz_dir = "/home/sam/Data_Compression/SZ3/tools/pysz"
if pysz_dir not in sys.path:
    sys.path.append(pysz_dir)
from pysz import SZ

def set_seed(seed=42):
    torch.manual_seed(seed)
    np.random.seed(seed)
    random.seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print(f"Device: {device} | GPUs: {torch.cuda.device_count()}")

In [ ]:
# ── Paths ─────────────────────────────────────────────────────────────────────
halo_finder_root = Path("/home/sam/Halo_Finder")
base_path   = (halo_finder_root / "halo_finder_v1/SDRBENCH-EXASKY-NYX-512x512x512/origin").as_posix() + "/"
sz_lib_path = "/home/sam/Data_Compression/SZ3/build/lib64/libSZ3c.so"
pysz_script = "/home/sam/Halo_Finder/halo_finder_v1/ROI_Compression/pysz.py"

# ── Dataset constants ──────────────────────────────────────────────────────────
data_shape   = (512, 512, 512)
FIELD_FILES  = [
     "baryon_density.f32", "temperature.f32", "dark_matter_density.f32" , "velocity_z.f32", 
     "velocity_x.f32", "velocity_y.f32",
]
TARGET_STEM  = "baryon_density"
TEST_REL_ERR = 1e-5

sz_engine = SZ(sz_lib_path)
print("SZ engine loaded")

In [ ]:
def rel_sz_suffix(rel_err):
    return f"{float(rel_err):.0e}".replace("+", "")

def load_nyx_field(target_stem, rel_err):
    fname      = f"{target_stem}.f32"
    gt_path    = base_path + fname
    aux_paths  = [base_path + f for f in FIELD_FILES if f != fname]
    sz_bin     = base_path + Path(fname).stem + "_rel" + rel_sz_suffix(rel_err) + ".sz"

    if not Path(sz_bin).is_file():
        vol = np.fromfile(gt_path, dtype=np.float32).reshape(data_shape)
        Path(sz_bin).write_bytes(sz_engine.compress(vol, 1, 0, float(rel_err), 0)[0])
        print(f"Created SZ binary: {sz_bin}")

    Xs_raw, _ = load_multifield_from_disk(
        gt_path=gt_path, aux_paths=aux_paths, sz_bin_path=sz_bin,
        data_shape=data_shape, pysz_path=pysz_script, sz_lib_path=sz_lib_path,
    )
    Xs         = [np.asarray(f, np.float32) for f in Xs_raw]
    aux_fields = [np.asarray(f, np.float32) for f in Xs_raw[1:]]

    def build_Xps_for_rel(target_rel_err):
        b, cr = sz_engine.compress(Xs[0], 1, 0, float(target_rel_err), 0)
        x_lq  = sz_engine.decompress(b, Xs[0].shape, np.float32)
        return [x_lq] + aux_fields, float(cr), int(len(b))

    return Xs, build_Xps_for_rel

In [ ]:
Xs, build_Xps_for_rel     = load_nyx_field(TARGET_STEM, TEST_REL_ERR)
Xps_list, sz_cr, sz_bytes = build_Xps_for_rel(TEST_REL_ERR)
Xps_list                  = [np.asarray(f, np.float32) for f in Xps_list]

orig_bytes = Xs[0].nbytes
print(f"Loaded {len(Xs)} fields | shape: {Xs[0].shape} | dtype: {Xs[0].dtype}")
print(f"SZ3 rel={TEST_REL_ERR:.0e} | CR={sz_cr:.2f}x | "
      f"original {orig_bytes/1e9:.2f} GB -> compressed {sz_bytes/1e6:.1f} MB")

In [ ]:
# Data layout: (Z, Y, X).
# Permute so the target slice axis becomes axis-0 (the "depth" axis the BG model
# iterates over); the remaining two axes are the 2-D spatial patch dims.
#
#   Z: (Z, Y, X) -> no change          axes (0,1,2)
#   Y: (Z, Y, X) -> (Y, Z, X)          axes (1,0,2)
#   X: (Z, Y, X) -> (X, Z, Y)          axes (2,0,1)
#
# Inverse permutations back to (Z, Y, X):
#   Z: (0,1,2)   Y: (1,0,2) [self-inv]   X: (1,2,0)

DIRECTIONS = ["Z", "Y", "X"]
_FWD = {"Z": (0, 1, 2), "Y": (1, 0, 2), "X": (2, 0, 1)}
_INV = {"Z": (0, 1, 2), "Y": (1, 0, 2), "X": (1, 2, 0)}


def permute_fields(fields, direction):
    axes = _FWD[direction]
    if axes == (0, 1, 2):
        return fields
    return [np.ascontiguousarray(np.transpose(f, axes)) for f in fields]


def unpermute_field(field, direction):
    axes = _INV[direction]
    if axes == (0, 1, 2):
        return field
    return np.ascontiguousarray(np.transpose(field, axes))


def downsample_spatial(fields, S=4):
    # Stride-downsample axes 1 and 2 by S; axis 0 (depth) stays intact.
    return [f[:, ::S, ::S] for f in fields]


def subsample_depth(fields, n_keep):
    # Keep `n_keep` evenly-spread FULL-RESOLUTION slices along axis 0.
    # This is the Phase-1 proxy: it preserves 512x512 patch statistics (so the
    # optimal LR transfers to Phase 2) while cutting per-epoch cost ~n/n_keep.
    n = fields[0].shape[0]
    idx = np.linspace(0, n - 1, min(n_keep, n)).round().astype(int)
    idx = np.unique(idx)
    return [np.ascontiguousarray(f[idx]) for f in fields]


# Roundtrip sanity check
for d in DIRECTIONS:
    pf  = permute_fields(Xs[:1], d)
    upf = unpermute_field(pf[0], d)
    assert np.allclose(Xs[0], upf), f"Roundtrip failed for direction {d}"
    sub = subsample_depth(pf, 32)
    print(f"  {d}: permuted {pf[0].shape}  ->  depth-subsampled {sub[0].shape}  roundtrip OK")

In [ ]:
def build_cfg(Xs_in, Xps_in, max_train_time, bg_h, steps_per_epoch,
              lr=1e-4, epochs=200, log_prefix="", patch_size=None):
    if patch_size is None:
        patch_size = Xs_in[0].shape[2]
    cfg = build_bg_only_cfg(
        X_target=Xs_in[0],
        Xps=Xps_in,
        max_train_time=max_train_time,
        epochs=epochs,
        steps_per_epoch=steps_per_epoch,
        bg_h=bg_h,
        bg_batch=1,
        bg_patch_size=patch_size,
        lr=lr,
    )
    cfg.bg_sample_mode   = "sequential"
    cfg.bg_log_prefix    = log_prefix
    cfg.bg_arch          = "spatial"
    cfg.amp              = True
    cfg.amp_dtype        = "bf16"
    cfg.bg_ddp           = False
    cfg.bg_data_parallel = False
    return cfg


def psnr_from_arrays(target, recon):
    data_range = float(target.max() - target.min())
    if data_range <= 0:
        data_range = 1.0
    mse = float(np.mean((target - recon) ** 2))
    return 100.0 if mse <= 0 else 20.0 * np.log10(data_range) - 10.0 * np.log10(mse)


print("Helpers ready")

## Phase 1 — Bayesian Optimization (Optuna TPE)

Search space (jointly): `lr ∈ [1e-4, 1e-2]` (log-uniform) × `direction ∈ {Z, Y, X}` (categorical).

**Why earlier runs only completed 2 trials:** the cost was not training but *data movement* — transposing the whole 512³ × 6-field volume inside every trial (~3 GB of copies), plus re-running `pick_bg_h_under_budget` and paying CUDA warm-up in trial #0. Fixes:

1. **`make_proxy`** — take the `TUNE_DEPTH` slices *first* (`np.take`), then transpose the tiny result; all 3 directions cached **once** before the timer starts
2. **`bg_h` computed once** (shape-identical across trials)
3. **GPU warm-up micro-trial** before the clock, so trial #0 isn't 3–4× slower
4. **Seed trials** (`enqueue_trial`): every direction × {3e-4, 3e-3} is tried first, guaranteeing all 3 directions appear in the study (and in the Phase-2 sweep) even if the timeout cuts the search short

Proxy = `TUNE_DEPTH` evenly-spread slices at S=2 (256×256) — close enough to full-res that the LR ranking transfers (S=4/128² shifts the LR optimum). **Objective** = PSNR at the *end* of the short trial. Budget enforced by `timeout = PHASE1_TIME` (≤ 20% of total).

In [ ]:
# ── Global time budget ────────────────────────────────────────────────────────
TOTAL_TIME  = 60.0                     # total wall-clock for the whole pipeline
PHASE1_FRAC = 0.1                    # Phase 1 may use at most 20% of total
PHASE1_TIME = TOTAL_TIME * PHASE1_FRAC # 12 s for Bayesian Optimization
PHASE2_TIME = TOTAL_TIME - PHASE1_TIME  # initial estimate; overwritten after Phase 1 completes

# ── Phase 1 proxy settings ────────────────────────────────────────────────────
# Goal: ~1 s/trial so >=10 trials fit in PHASE1_TIME=12 s.
#
# The previous version spent ~5 s/trial NOT on training but on data movement:
# permute_fields() transposed the whole 512^3 x 6-field volume (~3 GB of copies)
# inside every objective call.  Fix: slice FIRST (np.take of TUNE_DEPTH planes),
# THEN transpose the tiny result — and cache the proxy per direction up front.
TUNE_DEPTH        = 32                   # slices kept per trial
S_SPATIAL         = 2                   # spatial stride: 512 -> 256 (S=4 shifts the LR optimum)
TUNE_EPOCHS       = 999                 # effectively unlimited — training stops at PER_TRIAL_CAP
N_TRIALS          = 10                  # hard cap (keeps the Phase-2 sweep readable)
PARAM_BUDGET_TUNE = 30000
PER_TRIAL_CAP     = PHASE1_TIME / N_TRIALS  # evenly divide Phase-1 budget: 12s/10 = 1.2s/trial
FREQ_WARMUP       = 1                  # epochs before the FFT/freq loss term joins the total loss (default 3)

_DIR_AXIS = {"Z": 0, "Y": 1, "X": 2}

def make_proxy(fields, direction, n_keep=TUNE_DEPTH, S=S_SPATIAL):
    """Cheap proxy extraction: take n_keep planes along `direction` FIRST
    (copies only the selected slices), then permute the small array and
    stride-downsample its two spatial dims."""
    axis = _DIR_AXIS[direction]
    n    = fields[0].shape[axis]
    idx  = np.unique(np.linspace(0, n - 1, n_keep).round().astype(int))
    out  = []
    for f in fields:
        sub = np.take(f, idx, axis=axis)            # small copy: n_keep planes
        sub = np.transpose(sub, _FWD[direction])    # tiny transpose
        out.append(np.ascontiguousarray(sub[:, ::S, ::S]))
    return out

# Cache proxies for all 3 directions once (one-time data prep, ~1 s total)
t0 = time.time()
proxy_cache = {
    d: (make_proxy(Xs, d), make_proxy(Xps_list, d))
    for d in DIRECTIONS
}
proxy_shape = proxy_cache["Z"][0][0].shape
print(f"Proxy cache built in {time.time()-t0:.2f}s | per-direction shape: {proxy_shape}")

# bg_h is shape-dependent only — identical for all trials, compute ONCE
try:
    bg_h_tune, _ = pick_bg_h_under_budget(
        PARAM_BUDGET_TUNE, shape=proxy_shape,
        n_fields=len(proxy_cache["Z"][1]), bg_arch="spatial"
    )
    bg_h_tune = int(bg_h_tune)
except Exception:
    bg_h_tune = 10

print(f"Budget: total={TOTAL_TIME:.0f}s | Phase1 <= {PHASE1_TIME:.0f}s | Phase2 ~= {PHASE2_TIME:.0f}s")
print(f"Phase 1 proxy: {proxy_shape[0]} slices x {proxy_shape[1]}x{proxy_shape[2]} | "
      f"bg_h={bg_h_tune} | {N_TRIALS} trials x {PER_TRIAL_CAP:.2f}s/trial "
      f"= {N_TRIALS*PER_TRIAL_CAP:.1f}s (Phase-1 budget evenly split)")

all_tune_histories = {}   # (lr, direction) -> hist dict


def objective(trial):
    lr        = trial.suggest_float("lr", 1e-4, 1e-2, log=True)
    direction = trial.suggest_categorical("direction", DIRECTIONS)

    Xs_sub, Xps_sub = proxy_cache[direction]
    n_depth  = Xs_sub[0].shape[0]
    patch_sz = Xs_sub[0].shape[2]

    t_trial_start = time.time()
    tune_cfg = build_cfg(
        Xs_sub, Xps_sub,
        max_train_time=PER_TRIAL_CAP,   # hard per-trial wall-time cap
        bg_h=bg_h_tune,
        steps_per_epoch=n_depth,
        lr=lr,
        epochs=TUNE_EPOCHS,
        log_prefix=f"BO-{direction}-{lr:.1e}",
        patch_size=patch_sz,
    )
    tune_cfg.bg_freq_warmup_epochs = FREQ_WARMUP

    # Capture loop values via default args to avoid late-binding closure bug
    def evaluator(m, cfg=tune_cfg, Xs_d=Xs_sub, Xps_d=Xps_sub):
        m_core = unwrap_bg_model(m)
        xh     = run_bg_inference(m_core, Xs_d, Xps_d, cfg, TEST_REL_ERR)
        return psnr_from_arrays(Xs_d[0], xh), 0.0

    set_seed(42)
    _, hist = train_bg_only(
        Xs=Xs_sub, Xps=Xps_sub, device=device, cfg=tune_cfg, evaluator=evaluator
    )

    # Objective = FINAL PSNR (end of trial), a better long-horizon predictor
    psnr_vals   = [v[1] if isinstance(v, tuple) else v for v in hist.get("psnr", [])]
    final_psnr  = psnr_vals[-1] if psnr_vals else -1.0
    trial_secs  = time.time() - t_trial_start
    t_elapsed   = time.time() - phase1_start

    all_tune_histories[(lr, direction)] = hist
    print(f"  Trial {trial.number:2d}: lr={lr:.2e}  dir={direction}  "
          f"PSNR={final_psnr:.2f} dB  [{trial_secs:.1f}s/trial  {t_elapsed:.0f}s elapsed]")
    if trial_secs > PER_TRIAL_CAP * 2:
        print(f"    ⚠ trial took {trial_secs:.1f}s > cap {PER_TRIAL_CAP:.1f}s "
              f"— lower TUNE_DEPTH ({TUNE_DEPTH}) or raise S_SPATIAL ({S_SPATIAL})")
    return final_psnr


# ── GPU warm-up (one throwaway micro-trial, NOT counted in the budget) ────────
# Absorbs CUDA context / cuDNN autotune / first-call JIT cost so trial #0
# isn't 3-4x slower than the rest.
_warm_Xs, _warm_Xps = proxy_cache["Z"]
_warm_cfg = build_cfg(_warm_Xs, _warm_Xps, max_train_time=0.5, bg_h=bg_h_tune,
                      steps_per_epoch=2, lr=1e-3, epochs=1,
                      log_prefix="warmup", patch_size=_warm_Xs[0].shape[2])
set_seed(42)
train_bg_only(Xs=_warm_Xs, Xps=_warm_Xps, device=device, cfg=_warm_cfg,
              evaluator=lambda m, cfg=_warm_cfg: (0.0, 0.0))
torch.cuda.synchronize() if torch.cuda.is_available() else None
print("GPU warm-up done\n")

study = optuna.create_study(
    direction="maximize",
    sampler=optuna.samplers.TPESampler(seed=42, n_startup_trials=4),
)
# Seed trials: one per direction at a mid LR, so all 3 directions are covered
# (and appear in the Phase-2 sweep) even if the timeout cuts the search short.
# TPE then spends the remaining trials refining lr within the best direction(s).
for d in DIRECTIONS:
    study.enqueue_trial({"lr": 1e-3, "direction": d})

phase1_start = time.time()
# timeout enforces the <= 20% budget: Optuna stops launching new trials past it
study.optimize(objective, n_trials=N_TRIALS, timeout=PHASE1_TIME)

phase1_elapsed = time.time() - phase1_start
n_done         = len(study.trials)
best_lr        = study.best_params["lr"]
best_direction = study.best_params["direction"]
best_psnr_p1   = study.best_value

# Give any unused Phase-1 budget back to Phase 2
PHASE2_TIME = TOTAL_TIME - phase1_elapsed

dirs_tried = {t.params["direction"] for t in study.trials}
print(f"\nPhase 1 done in {phase1_elapsed:.1f}s ({n_done} trials, "
      f"{phase1_elapsed/TOTAL_TIME*100:.0f}% of total budget)")
print(f"Phase 2 budget: {PHASE2_TIME:.1f}s  "
      f"(={TOTAL_TIME:.0f}s total − {phase1_elapsed:.1f}s used by Phase 1)")
print(f"Directions covered: {sorted(dirs_tried)}")
if n_done < 8:
    print(f"  ⚠ Only {n_done} trials — lower TUNE_DEPTH ({TUNE_DEPTH}) or raise S_SPATIAL ({S_SPATIAL})")
print(f"Best (Phase-1 proxy): lr={best_lr:.2e}  direction={best_direction}  "
      f"PSNR={best_psnr_p1:.2f} dB")

In [ ]:
dir_colors  = {"Z": "#1f77b4", "Y": "#ff7f0e", "X": "#2ca02c"}
dir_markers = {"Z": "o",       "Y": "s",        "X": "^"}

sorted_trials = sorted(study.trials, key=lambda t: -t.value)
rows = [
    [str(t.number), f"{t.params['lr']:.2e}", t.params["direction"], f"{t.value:.2f} dB"]
    for t in sorted_trials
]

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Left: ranked table
axes[0].axis("off")
tbl = axes[0].table(
    cellText=rows,
    colLabels=["Trial", "LR", "Direction", "PSNR"],
    loc="center", cellLoc="center",
)
tbl.scale(1.15, 1.7)
tbl.auto_set_font_size(False)
tbl.set_fontsize(11)
for j in range(4):
    tbl[1, j].set_facecolor("#d4f0a0")
axes[0].set_title(
    f"Phase 1 Results (ranked)\nBest: dir={best_direction}, lr={best_lr:.2e}",
    fontsize=12, pad=20
)

# Right: scatter LR vs PSNR, coloured by direction; red star = BO pick
seen_dirs = set()
for t in study.trials:
    d   = t.params["direction"]
    lbl = f"dir={d}" if d not in seen_dirs else "_nolegend_"
    seen_dirs.add(d)
    axes[1].scatter(
        t.params["lr"], t.value,
        c=dir_colors[d], marker=dir_markers[d], s=90, zorder=3, label=lbl
    )
axes[1].scatter(best_lr, best_psnr_p1, c="red", s=260, marker="*", zorder=5,
                label=f"BO pick ({best_direction}, {best_lr:.1e})")
axes[1].set_xscale("log")
axes[1].set_xlabel("Learning Rate")
axes[1].set_ylabel("Final PSNR (dB)  [proxy]")
axes[1].set_title(f"BO Search — NYX {TARGET_STEM} (rel={TEST_REL_ERR:.0e})\n"
                  f"red star = BO pick (lr*, d*)")
axes[1].grid(True, alpha=0.3)
axes[1].legend()

plt.tight_layout()
plt.savefig("phase1_bo_results.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved: phase1_bo_results.png")

In [ ]:
from scipy.stats import gaussian_kde

GAMMA = 0.25

all_vals = sorted(
    [(t.value, t.params["lr"], t.params["direction"]) for t in study.trials],
    reverse=True
)
n_good = max(1, int(len(all_vals) * GAMMA))
good_trials_tpe = all_vals[:n_good]
bad_trials_tpe  = all_vals[n_good:]

psnr_vals  = [t.value        for t in study.trials]
lr_vals    = [t.params["lr"] for t in study.trials]
dir_vals   = [t.params["direction"] for t in study.trials]
trial_nums = [t.number       for t in study.trials]

dir_y      = {"Z": 0, "Y": 1, "X": 2}
lr_range   = np.logspace(np.log10(1e-4), np.log10(1e-2), 300)
log_lr_range = np.log(lr_range)

# ═══════════════════════════════════════════════════════════════════════════════
# Figure A — Direction sampling weights  |  Optimisation history  |  Search map
# ═══════════════════════════════════════════════════════════════════════════════
figA, axA = plt.subplots(1, 3, figsize=(20, 5))
figA.suptitle("TPE Interpretability — Part 1: Search Overview", fontsize=14, fontweight="bold")

# ── A0: Direction good/bad frequency (manual importance, no sklearn) ──────────
dir_order = ["Z", "Y", "X"]
good_dir_cnt = {d: sum(1 for _, _, dir_ in good_trials_tpe if dir_ == d) for d in dir_order}
bad_dir_cnt  = {d: sum(1 for _, _, dir_ in bad_trials_tpe  if dir_ == d) for d in dir_order}
x_pos = np.arange(len(dir_order))
w = 0.35
bars_g = axA[0].bar(x_pos - w/2, [good_dir_cnt[d] for d in dir_order], w,
                    color=["#2ca02c","#ff7f0e","#1f77b4"], alpha=0.85,
                    label=f"top {n_good} (good)")
bars_b = axA[0].bar(x_pos + w/2, [bad_dir_cnt[d]  for d in dir_order], w,
                    color=["#2ca02c","#ff7f0e","#1f77b4"], alpha=0.35,
                    label="rest (bad)", hatch="//")
for bar in list(bars_g) + list(bars_b):
    h = bar.get_height()
    if h:
        axA[0].text(bar.get_x() + bar.get_width()/2, h + 0.05, str(int(h)),
                    ha="center", va="bottom", fontsize=10)
axA[0].set_xticks(x_pos); axA[0].set_xticklabels(dir_order, fontsize=12)
axA[0].set_ylabel("# Trials")
axA[0].set_title("Direction Frequency in Good vs Bad Trials" "(tall solid bar = TPE favours this direction)")
axA[0].legend(fontsize=9); axA[0].grid(axis="y", alpha=0.3)
axA[0].set_ylim(0, max(list(good_dir_cnt.values()) + list(bad_dir_cnt.values())) + 1.5)

# ── A1: Optimisation history ──────────────────────────────────────────────────
best_so_far = [max(psnr_vals[:i+1]) for i in range(len(psnr_vals))]
axA[1].plot(trial_nums, psnr_vals, "o--", alpha=0.45, color="steelblue", label="trial PSNR")
axA[1].plot(trial_nums, best_so_far, "r-", linewidth=2.5, label="best so far")
for num, psnr, d in zip(trial_nums, psnr_vals, dir_vals):
    axA[1].annotate(d, (num, psnr),
                    textcoords="offset points", xytext=(0, 7),
                    ha="center", fontsize=9, color=dir_colors[d], fontweight="bold")
axA[1].set_xlabel("Trial Number", fontsize=11)
axA[1].set_ylabel("Proxy PSNR (dB)", fontsize=11)
axA[1].set_title("Optimisation History (red = running best; letter = direction chosen)")
axA[1].legend(fontsize=9); axA[1].grid(True, alpha=0.3)

# ── A2: (lr, direction) 2-D search map ───────────────────────────────────────
norm_c = plt.Normalize(min(psnr_vals), max(psnr_vals))
sc = axA[2].scatter(
    lr_vals, [dir_y[d] for d in dir_vals],
    c=psnr_vals, cmap="RdYlGn", norm=norm_c,
    s=260, zorder=3, edgecolors="k", linewidths=0.6
)
for num, lr_, d in zip(trial_nums, lr_vals, dir_vals):
    axA[2].annotate(str(num), (lr_, dir_y[d]),
                    ha="center", va="center", fontsize=8, fontweight="bold")
figA.colorbar(sc, ax=axA[2], label="Proxy PSNR (dB)", pad=0.02)
axA[2].set_xscale("log")
axA[2].set_yticks([0, 1, 2]); axA[2].set_yticklabels(["Z", "Y", "X"], fontsize=12)
axA[2].axhline(dir_y[best_direction], color="red", linestyle="--", alpha=0.45, linewidth=1.2)
axA[2].axvline(best_lr,               color="red", linestyle="--", alpha=0.45, linewidth=1.2)
axA[2].scatter([best_lr], [dir_y[best_direction]], s=350, marker="*",
               color="red", zorder=5, label="BO pick")
axA[2].set_xlabel("Learning Rate (log scale)", fontsize=11)
axA[2].set_ylabel("Slice Direction", fontsize=11)
axA[2].set_title("Search Space Map (number = trial order · colour = PSNR · ★ = BO pick)")
axA[2].legend(fontsize=9); axA[2].grid(True, alpha=0.2)

figA.tight_layout()
figA.savefig("interp_A_overview.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved: interp_A_overview.png")

# ═══════════════════════════════════════════════════════════════════════════════
# Figure B — TPE l(x)/g(x) KDE per direction  (one subplot per direction)
# ═══════════════════════════════════════════════════════════════════════════════
figB, axB = plt.subplots(1, 3, figsize=(20, 5), sharey=False)
figB.suptitle(
    "TPE Interpretability — Part 2: KDE of LR in Good vs Bad Trials (per direction)\n"
    "Green l(x) = LR distribution of good trials  ·  Red g(x) = bad trials  ·  "
    "Black dashed = l/g ratio (TPE samples where this peaks)",
    fontsize=12, fontweight="bold"
)

_dir_col = {"Z": "#1f77b4", "Y": "#ff7f0e", "X": "#2ca02c"}

for i, d in enumerate(["Z", "Y", "X"]):
    ax = axB[i]
    good_lrs = [lr for _, lr, dir_ in good_trials_tpe if dir_ == d]
    bad_lrs  = [lr for _, lr, dir_ in bad_trials_tpe  if dir_ == d]
    all_lrs_d = [lr for _, lr, dir_ in all_vals       if dir_ == d]

    # rug plot: actual sampled LR values
    for lr_ in all_lrs_d:
        ax.axvline(lr_, color=_dir_col[d], alpha=0.20, linewidth=1.2, ymin=0, ymax=0.06)

    g_dens = b_dens = None
    if len(good_lrs) >= 2:
        kde_g = gaussian_kde(np.log(good_lrs), bw_method=0.5)
        g_dens = kde_g(log_lr_range)
        ax.fill_between(lr_range, g_dens, alpha=0.60, color="green",
                        label=f"l(x) — good (top {n_good})")
        ax.plot(lr_range, g_dens, color="darkgreen", linewidth=1.5)
    elif len(good_lrs) == 1:
        ax.axvline(good_lrs[0], color="green", linewidth=2.5,
                   label=f"l(x) — 1 good trial  (lr={good_lrs[0]:.1e})")

    if len(bad_lrs) >= 2:
        kde_b = gaussian_kde(np.log(bad_lrs), bw_method=0.5)
        b_dens = kde_b(log_lr_range)
        ax.fill_between(lr_range, b_dens, alpha=0.35, color="red",
                        label=f"g(x) — bad ({len(bad_lrs)} trials)")
        ax.plot(lr_range, b_dens, color="darkred", linewidth=1.5)
    elif len(bad_lrs) == 1:
        ax.axvline(bad_lrs[0], color="red", linestyle="--", linewidth=1.5, alpha=0.8,
                   label=f"g(x) — 1 bad trial  (lr={bad_lrs[0]:.1e})")

    # l/g ratio on twin axis (only when both KDEs exist)
    if g_dens is not None and b_dens is not None:
        ratio = np.where(b_dens > 1e-6, g_dens / b_dens, 0.0)
        ax2 = ax.twinx()
        ax2.plot(lr_range, ratio / (ratio.max() + 1e-9), "k--",
                 linewidth=2, alpha=0.75, label="l/g ratio (∝ EI)")
        ax2.set_ylabel("l(x)/g(x)  [normalised]", fontsize=9)
        ax2.set_ylim(0, 1.4)
        ax2.legend(loc="upper left", fontsize=8)

    # BO pick marker (only for the winning direction)
    if d == best_direction:
        ax.axvline(best_lr, color="red", linewidth=2.2, linestyle="-",
                   label=f"BO pick  lr={best_lr:.1e} ★")

    # annotation if no trials at all
    if not all_lrs_d:
        ax.text(0.5, 0.5, "No trials sampled for this direction",
                ha="center", va="center", transform=ax.transAxes,
                fontsize=11, color="gray")

    ax.set_xscale("log")
    ax.set_xlabel("Learning Rate", fontsize=11)
    ax.set_ylabel("Density", fontsize=11)
    ax.set_title(f"Direction = {d}   ({len(all_lrs_d)} trials sampled)", fontsize=12)
    ax.legend(fontsize=9, loc="upper center")
    ax.grid(True, alpha=0.3)

figB.tight_layout()
figB.savefig("interp_B_kde.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved: interp_B_kde.png")

## Phase 2 — Full-Resolution Validation

**Production use:** trust Phase-1's BO pick `(best_lr, best_direction)` directly and train it on the full volume for the remaining ~48 s. That's the whole pipeline.

**This notebook (validation mode):** to *check* that the BO pick is actually good, we re-run **every** `(lr, direction)` combo Phase 1 tried on the **full original (Z,Y,X)** volume for `PHASE2_TIME` each, and plot all PSNR-vs-time curves. The test: does the BO proxy pick (red star) land at/near the true full-resolution best? If yes, the cheap Phase-1 proxy is trustworthy and no extra validation is needed in production.

In [ ]:
# Reload bg_stage so re-running ONLY this cell (without the import cell) still
# picks up early-stop edits.  The assertion fails LOUDLY on a stale module
# instead of silently running old behavior (the cause of "still stops at 11").
import importlib, inspect, bg_stage
importlib.reload(bg_stage)
from bg_stage import run_bg_inference, train_bg_only, unwrap_bg_model
assert "loss_patience" in inspect.getsource(train_bg_only), (
    "STALE bg_stage loaded -> Kernel > Restart, then Run All.")
print(f"bg_stage OK: {bg_stage.__file__}")

PARAM_BUDGET_P2 = 30000

# ── Run ALL (lr, direction) combos tried in Phase 1 ──────────────────────────
# Sort by descending proxy PSNR so the best candidates run first
all_p2_configs = sorted(
    all_tune_histories.keys(),
    key=lambda c: -(
        [v[1] if isinstance(v, tuple) else v
         for v in all_tune_histories[c].get("psnr", []) or [(-1,)]][-1]
    )
)

est_total = len(all_p2_configs) * PHASE2_TIME
print(f"Phase 2: running all {len(all_p2_configs)} Phase-1 configs | "
      f"each for {PHASE2_TIME:.0f}s (~{est_total/60:.0f} min total)")
for i, (lr, d) in enumerate(all_p2_configs):
    tag = "  <- BO #1" if (lr == best_lr and d == best_direction) else ""
    print(f"  [{i+1:2d}] lr={lr:.2e}  dir={d}{tag}")

# bg_h depends only on shape — identical for all directions of a cube volume
try:
    bg_h_p2, _ = pick_bg_h_under_budget(
        PARAM_BUDGET_P2, shape=Xs[0].shape, n_fields=len(Xps_list),
        bg_arch="spatial"
    )
    bg_h_p2 = int(bg_h_p2)
except Exception:
    bg_h_p2 = 10
print(f"bg_h = {bg_h_p2}")

full_histories = {}   # (lr, direction) -> hist

# Group configs by direction so the expensive full-volume permute (~3 GB of
# copies) happens once per direction instead of once per config
for direction_p2 in DIRECTIONS:
    lrs_this_dir = [lr for lr, d in all_p2_configs if d == direction_p2]
    if not lrs_this_dir:
        continue

    print(f"\n{'#'*55}")
    print(f"# Direction {direction_p2}: {len(lrs_this_dir)} LR configs")
    t0 = time.time()
    Xs_perm  = permute_fields(Xs, direction_p2)
    Xps_perm = permute_fields(Xps_list, direction_p2)
    n_depth  = Xs_perm[0].shape[0]
    patch_sz = Xs_perm[0].shape[2]
    print(f"# permute done in {time.time()-t0:.1f}s")

    for lr_p2 in lrs_this_dir:
        is_pick = (lr_p2 == best_lr and direction_p2 == best_direction)
        tag = "BO #1" if is_pick else "trial"
        print(f"\n{'='*55}")
        print(f"[{tag}]  lr={lr_p2:.2e}  dir={direction_p2}  budget={PHASE2_TIME:.0f}s")

        p2_cfg = build_cfg(
            Xs_perm, Xps_perm,
            max_train_time=PHASE2_TIME,
            bg_h=bg_h_p2,
            steps_per_epoch=n_depth,
            lr=lr_p2,
            epochs=200,
            log_prefix=f"P2-{direction_p2}-{lr_p2:.1e}",
            patch_size=patch_sz,
        )
        # Slope-based early stop on the *training loss* (computed for free each
        # epoch — no extra full-volume inference).  Conservative: only stops once
        # the loss has genuinely flattened, so it won't cut a good config short.
        p2_cfg.bg_early_stop   = True
        p2_cfg.bg_es_metric    = "loss_patience"  # relative training loss (no eval overhead)
        p2_cfg.bg_es_min_drop  = 0.02     # 2% per-epoch relative drop threshold
        p2_cfg.bg_es_patience  = 2        # <2% drop for 2 consecutive epochs -> stop
        p2_cfg.bg_freq_warmup_epochs = FREQ_WARMUP  # freq loss term joins at ep >= this

        # Default-arg capture to avoid late-binding closure across loop iterations
        def evaluator_p2(m, cfg=p2_cfg, Xs_p=Xs_perm, Xps_p=Xps_perm, dir_=direction_p2):
            m_core  = unwrap_bg_model(m)
            xh_perm = run_bg_inference(m_core, Xs_p, Xps_p, cfg, TEST_REL_ERR)
            xh      = unpermute_field(xh_perm, dir_)   # back to (Z, Y, X)
            return psnr_from_arrays(Xs[0], xh), 0.0

        set_seed(42)
        _, hist = train_bg_only(
            Xs=Xs_perm, Xps=Xps_perm, device=device, cfg=p2_cfg, evaluator=evaluator_p2
        )
        full_histories[(lr_p2, direction_p2)] = hist

    # Free the permuted copies before moving to the next direction
    del Xs_perm, Xps_perm

print("\n--- Phase 2 complete ---")

# ── Validation: is the BO proxy pick close to the true full-res best? ─────────
def final_of(hist):
    ps = [v[1] if isinstance(v, tuple) else v for v in hist.get("psnr", [])]
    return ps[-1] if ps else None

final_psnr   = {c: final_of(h) for c, h in full_histories.items()}
final_psnr   = {c: v for c, v in final_psnr.items() if v is not None}
final_winner = max(final_psnr, key=final_psnr.get) if final_psnr else None
bo_pick      = (best_lr, best_direction)

print(f"\nBO proxy pick : lr={bo_pick[0]:.2e}, dir={bo_pick[1]} "
      f"({final_psnr.get(bo_pick, float('nan')):.2f} dB at full res)")
print(f"True best     : lr={final_winner[0]:.2e}, dir={final_winner[1]} "
      f"({final_psnr[final_winner]:.2f} dB)")
gap = final_psnr[final_winner] - final_psnr.get(bo_pick, final_psnr[final_winner])
if bo_pick == final_winner:
    print("  ✅ BO proxy pick IS the true full-res best.")
elif gap < 0.5:
    print(f"  ✅ BO pick within {gap:.2f} dB of the best — proxy is trustworthy.")
else:
    print(f"  ⚠ BO pick trails the best by {gap:.2f} dB — proxy may need a finer search.")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(24, 6))

# ── Left: Phase 1 — PSNR vs wall-time for every proxy trial ──────────────────
all_lrs_p1_sorted = sorted(set(lr for lr, _ in all_tune_histories))
_ls_p1 = ["-", "--", "-.", ":", (0,(3,1,1,1)), (0,(5,1))]
lr_ls_p1 = {lr: _ls_p1[i % len(_ls_p1)] for i, lr in enumerate(all_lrs_p1_sorted)}

# time offset: each trial starts where the previous ended
t_offset = 0.0
for t in sorted(study.trials, key=lambda t: t.number):
    lr_t  = t.params["lr"]
    d_t   = t.params["direction"]
    hist  = all_tune_histories.get((lr_t, d_t), {})
    t_raw = hist.get("time", [])
    p_raw = [v[1] if isinstance(v, tuple) else v for v in hist.get("psnr", [])]
    if not t_raw or not p_raw:
        t_offset += PER_TRIAL_CAP
        continue

    t_abs = [t_offset + tt for tt in t_raw]
    is_pick = (lr_t == best_lr and d_t == best_direction)
    lw    = 2.8 if is_pick else 1.0
    alpha = 1.0 if is_pick else 0.55
    zo    = 5   if is_pick else 2
    suffix = " ★BO pick" if is_pick else ""
    label  = f"lr={lr_t:.1e}, d={d_t}{suffix}"

    axes[0].plot(t_abs, p_raw, linestyle=lr_ls_p1[lr_t], linewidth=lw,
                 alpha=alpha, color=dir_colors[d_t], zorder=zo, label=label,
                 marker=dir_markers[d_t] if is_pick else None, markersize=5)
    axes[0].annotate(f"{p_raw[-1]:.1f}",
                     xy=(t_abs[-1], p_raw[-1]),
                     xytext=(3, 0), textcoords="offset points",
                     fontsize=7, color=dir_colors[d_t],
                     fontweight="bold" if is_pick else "normal", va="center")
    t_offset = t_abs[-1]

axes[0].set_xlabel("Wall Time (s)")
axes[0].set_ylabel("Proxy PSNR (dB)")
axes[0].set_title(
    f"Phase 1: BO ({n_done} trials · {PER_TRIAL_CAP:.2f}s/trial · "
    f"{TUNE_DEPTH} slices × {data_shape[1]//S_SPATIAL}×{data_shape[2]//S_SPATIAL} · {phase1_elapsed:.1f}s total)\n"
    f"★ = BO pick (lr*, d*)"
)
axes[0].grid(True, alpha=0.35)
axes[0].legend(fontsize=7, ncol=2)

# ── Right: ALL Phase-1 (lr, direction) combos — PSNR vs wall time ────────────
def _curve(hist):
    t = hist.get("time", [])
    p = [v[1] if isinstance(v, tuple) else v for v in hist.get("psnr", [])]
    return t, p

# Assign a unique linestyle per distinct lr value so curves with the same
# direction (= same colour) can still be told apart
all_lrs_sorted = sorted(set(lr for lr, _ in full_histories))
_linestyles    = ["-", "--", "-.", ":", (0,(3,1,1,1)), (0,(5,1))]
lr_linestyle   = {lr: _linestyles[i % len(_linestyles)]
                  for i, lr in enumerate(all_lrs_sorted)}

for (lr, d), hist in sorted(full_histories.items(),
                             key=lambda kv: (kv[0][1], kv[0][0])):
    t_vals, p_vals = _curve(hist)
    if not t_vals or not p_vals:
        continue

    is_best = ((lr, d) == final_winner)            # true full-res best
    is_pick = ((lr, d) == (best_lr, best_direction))  # BO's choice
    color   = dir_colors[d]
    ls      = lr_linestyle[lr]
    lw      = 3.2 if is_pick else (2.4 if is_best else 1.0)
    alpha   = 1.0 if (is_pick or is_best) else 0.6
    zorder  = 5 if is_pick else (4 if is_best else 2)

    suffix = ""
    if is_pick: suffix += " ★BO pick"
    if is_best: suffix += " (true best)"
    label = f"lr={lr:.1e}, d={d}{suffix}"

    axes[1].plot(t_vals, p_vals, linestyle=ls, linewidth=lw, alpha=alpha,
                 color=color, zorder=zorder, label=label,
                 marker=dir_markers[d] if is_pick else None,
                 markersize=5)

    # Annotate final PSNR value at the end of every curve
    axes[1].annotate(
        f"{p_vals[-1]:.1f}",
        xy=(t_vals[-1], p_vals[-1]),
        xytext=(3, 0), textcoords="offset points",
        fontsize=7, color=color,
        fontweight="bold" if is_pick else "normal",
        va="center",
    )

gap = final_psnr[final_winner] - final_psnr.get((best_lr, best_direction),
                                                final_psnr[final_winner])
verdict = ("BO pick = true best ✓" if (best_lr, best_direction) == final_winner
           else f"BO pick within {gap:.2f} dB of best")
axes[1].set_xlabel("Wall Time (s)")
axes[1].set_ylabel("Global PSNR (dB)  [full (Z,Y,X) volume]")
axes[1].set_title(
    f"Phase 2: All {len(full_histories)} Phase-1 configs ({PHASE2_TIME:.0f}s each)\n"
    f"{verdict}"
)
axes[1].grid(True, alpha=0.3)
# Legend outside the axes so curves aren't covered
axes[1].legend(fontsize=7, ncol=2, loc="upper left",
               bbox_to_anchor=(1.01, 1), borderaxespad=0)

# ── axes[2]: Loss vs epoch for the BO-selected config (drives early-stop) ─────
bo_hist   = full_histories.get((best_lr, best_direction), {})
bo_loss   = list(bo_hist.get("loss", []))
bo_epochs = list(bo_hist.get("epoch", list(range(1, len(bo_loss) + 1))))
# history["epoch"] carries an epoch-0 init point (PSNR only, no loss), so it
# can be one longer than the loss list — align to the loss tail before plotting.
if len(bo_epochs) != len(bo_loss):
    n = min(len(bo_epochs), len(bo_loss))
    bo_epochs, bo_loss = bo_epochs[-n:], bo_loss[-n:]
if bo_loss:
    axes[2].plot(bo_epochs, bo_loss, "-o", color=dir_colors[best_direction],
                 markersize=4, linewidth=1.8,
                 label=f"lr={best_lr:.1e}, d={best_direction}")
    axes[2].scatter([bo_epochs[-1]], [bo_loss[-1]], s=170, marker="*",
                    color="red", zorder=5, label=f"stop @ epoch {bo_epochs[-1]}")
    axes[2].annotate(f"{bo_loss[-1]:.2e}", xy=(bo_epochs[-1], bo_loss[-1]),
                     xytext=(5, 0), textcoords="offset points",
                     fontsize=8, color="red", va="center")
    axes[2].set_yscale("log")
    # Mark where the frequency-loss term switches on (bg_freq_warmup_epochs).
    # The TOTAL loss steps UP once here because a new term enters the sum —
    # this is a loss-composition change, NOT divergence (PSNR keeps improving).
    _fw = FREQ_WARMUP   # freq loss term added at ep >= _fw (Epoch _fw+1)
    if bo_epochs and bo_epochs[-1] > _fw:
        axes[2].axvline(_fw + 0.5, color="gray", linestyle="--", linewidth=1.2, alpha=0.7)
        axes[2].text(_fw + 0.5, 0.96, f"  freq loss term on (warmup={_fw})",
                     transform=axes[2].get_xaxis_transform(), rotation=90,
                     fontsize=7, color="gray", va="top", ha="left")
else:
    axes[2].text(0.5, 0.5, "no loss history for BO pick",
                 ha="center", va="center", transform=axes[2].transAxes)
axes[2].set_xlabel("Epoch")
axes[2].set_ylabel("Training Loss  (log scale)")
axes[2].set_title(f"Phase 2 loss curve — BO pick (lr={best_lr:.1e}, d={best_direction})  |  "
                  f"early-stop: loss drop < 2% for 2 consecutive epochs")
axes[2].grid(True, alpha=0.3, which="both")
axes[2].legend(fontsize=8)

plt.suptitle(
    f"Two-Phase BO — NYX {TARGET_STEM}  rel={TEST_REL_ERR:.0e}  SZ3 CR={sz_cr:.2f}x",
    fontsize=14
)
plt.tight_layout()
out_png = f"BO_lr_direction_NYX_{TARGET_STEM}_rel{TEST_REL_ERR:.0e}_{PHASE1_FRAC * 100}%.png"
plt.savefig(out_png, dpi=150, bbox_inches="tight")
plt.show()
print(f"Saved: {out_png}")

# ── Summary ───────────────────────────────────────────────────────────────────
print("\n" + "="*60)
print(f"SUMMARY — NYX {TARGET_STEM}  rel={TEST_REL_ERR:.0e}")
print(f"  Phase 1 (BO)      : {phase1_elapsed:.1f}s, {n_done} trials "
      f"({phase1_elapsed/TOTAL_TIME*100:.0f}% of budget)")
print(f"  >> BO pick (USE)  : lr={best_lr:.2e}, dir={best_direction}  "
      f"({final_psnr.get((best_lr, best_direction), float('nan')):.2f} dB at full res)")
print(f"  True best (ref)   : lr={final_winner[0]:.2e}, dir={final_winner[1]}  "
      f"({final_psnr[final_winner]:.2f} dB)")
print(f"\n  Phase-2 final PSNR (full volume), ranked:")
for cfg, v in sorted(final_psnr.items(), key=lambda kv: -kv[1]):
    lr, d = cfg
    tags = []
    if cfg == (best_lr, best_direction): tags.append("BO pick")
    if cfg == final_winner:              tags.append("true best")
    tag = ("  <- " + ", ".join(tags)) if tags else ""
    print(f"    lr={lr:.2e}  dir={d}  ->  {v:.2f} dB{tag}")

# Verdict: is the cheap Phase-1 proxy trustworthy?
gap = final_psnr[final_winner] - final_psnr.get((best_lr, best_direction),
                                                final_psnr[final_winner])
if (best_lr, best_direction) == final_winner:
    print(f"\n  ✅ BO pick IS the true full-res best — proxy is reliable, "
          f"no extra validation needed in production.")
elif gap < 0.5:
    print(f"\n  ✅ BO pick within {gap:.2f} dB of the best — proxy is trustworthy.")
else:
    print(f"\n  ⚠ BO pick trails the best by {gap:.2f} dB — consider more BO trials "
          f"or a larger proxy (TUNE_DEPTH / S_SPATIAL).")